# 基于Agent轨迹的SFT：全量数据训练

## 1. 全量训练配置与下载原始数据

In [1]:
from pathlib import Path
from collections import Counter
from copy import deepcopy
from typing import Any
import json
import random
import re
import gc

import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

# 运行参数：保留先期验证时已成功训练的设置
DATA_DIR = Path("./sft-data")
OUTPUT_DIR = Path("./sft-output/gemma4-e2b-lora-full")
DATASET_ID = "badlogicgames/pi-mono"
MODEL_ID = "google/gemma-4-E2B-it"
TOKENIZER_DIR = DATA_DIR / "gemma4-tokenizer"

MAX_LENGTH = 2048
EVAL_RATIO = 0.10
SEED = 42

TRAIN_CONFIG = {
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "num_train_epochs": 3,
    "max_grad_norm": 1.0,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)

print("训练模型:", MODEL_ID)
print("最大序列长度:", MAX_LENGTH)
print("训练输出目录:", OUTPUT_DIR)

训练模型: google/gemma-4-E2B-it
最大序列长度: 2048
训练输出目录: sft-output/gemma4-e2b-lora-full


In [2]:
# 下载完整数据集（若本地已有文件则直接复用）
def download_dataset(data_dir: Path, dataset_id: str) -> Path:
    raw_dir = data_dir / "raw"
    if not list(raw_dir.rglob("*.jsonl")):
        snapshot_download(
            repo_id=dataset_id,
            repo_type="dataset",
            allow_patterns=["*.jsonl"],
            local_dir=str(raw_dir),
        )
    return raw_dir

raw_dir = download_dataset(DATA_DIR, DATASET_ID)
raw_files = sorted(raw_dir.rglob("*.jsonl"))
if not raw_files:
    raise FileNotFoundError(f"没有找到 JSONL 文件：{raw_dir}")

print("原始 Session 文件数:", len(raw_files))

原始 Session 文件数: 634


## 2. 读取Agent事件流与恢复对话分支
每个 JSONL 是一个 Session 的事件流；如果存在由 parentId 形成的分支，默认选择文件中最后一条消息所在的祖先路径，避免把不同分支混在同一段上下文中。无可靠父子链接时使用文件顺序。该策略相当于每个 Session 只选一个最终分支，不会把所有备选分支都当作独立训练轨迹。

In [3]:
# 读取单个 Session 的事件
def load_session(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8", errors="replace") as f:
        return [json.loads(line) for line in f if line.strip()]

# 优先按 parentId 取最后一个消息节点的祖先链，保留事件顺序。
def restore_message_branch(events: list[dict]) -> list[dict]:
    message_events = [e for e in events if e.get("type") == "message"]
    if not message_events:
        return []

    event_map = {e["id"]: e for e in events if e.get("id")}
    linked = sum(e.get("parentId") in event_map for e in message_events)

    # 某些文件只有顺序记录，没有可用的父节点结构
    if linked < max(1, len(message_events) // 2):
        return message_events

    leaf = next((e for e in reversed(message_events) if e.get("id")), None)
    if leaf is None:
        return message_events

    chain = []
    visited = set()
    current = leaf
    while current is not None:
        current_id = current.get("id")
        if not current_id or current_id in visited:
            break
        visited.add(current_id)
        if current.get("type") == "message":
            chain.append(current)
        current = event_map.get(current.get("parentId"))

    chain.reverse()
    return chain

## 3. 将Agent消息转换成标准对话格式
保留 `user`、`assistant`、`tool`；Assistant 的 `tool_calls` 与 Tool 的 `tool_call_id` 用于表达工具调用关系。原始 Thinking 不作为监督文本；图片以占位符标记（本 Notebook 不训练视觉输入）。

In [4]:
# 提取文本内容，暂不处理thinking推理部分
def extract_text_content(content):
    if isinstance(content, str):
        return content.strip()

    if not isinstance(content, list):
        return ""

    texts = []

    for block in content:
        if not isinstance(block, dict):
            continue

        if block.get("type") == "text":
            texts.append(block.get("text", ""))

        elif block.get("type") == "image":
            texts.append("[image omitted]")

    return "\n".join(texts).strip()

In [5]:
# 转换工具调用
def convert_tool_call(block):
    if not block.get("name") or not block.get("id"):
        return None

    return {
        "id": block["id"],
        "type": "function",
        "function": {
            "name": block["name"],
            "arguments": block.get("arguments") or {}
        }
    }

In [6]:
# 将原始User/Assistant/Tool消息统一转换成标注对话格式
def convert_event_to_message(event):
    # 跳过非消息事件
    if event.get("type") != "message":
        return None

    raw_message = event.get("message") or {}
    role = raw_message.get("role")
    content = raw_message.get("content", [])

    # User 消息
    if role == "user":
        text = extract_text_content(content)
        if not text:
            return None

        return {
            "role": "user",
            "content": text
        }

    # Assistant 消息
    elif role == "assistant":
        text = extract_text_content(content)

        tool_calls = []
        for block in content if isinstance(content, list) else []:
            if isinstance(block, dict) and block.get("type") == "toolCall":
                call = convert_tool_call(block)
                if call is not None:
                    tool_calls.append(call)

        if not text and not tool_calls:
            return None

        message = {
            "role": "assistant",
            "content": text
        }

        if tool_calls:
            message["tool_calls"] = tool_calls

        return message

    # Tool Result 消息
    elif role == "toolResult":
        tool_call_id = raw_message.get("toolCallId")
        if not tool_call_id:
            return None

        return {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "name": raw_message.get("toolName", "unknown"),
            "content": extract_text_content(content) or "[empty tool result]"
        }

    return None

In [7]:
# 将一个 Session 的事件转换为对话消息
def convert_session_messages(events: list[dict]) -> list[dict]:
    messages = []
    for event in restore_message_branch(events):
        message = convert_event_to_message(event)
        if message is not None:
            messages.append(message)
    return messages

## 4. 构建 SFT 监督样本

对于历史中的每条 Assistant 消息，使用它之前的消息作为 Prompt、当前 Assistant 消息作为 Completion。为了减少内存占用，此处**逐条生成样本**，而不是把全部历史和所有样本一次性保存在内存中。

In [8]:
# 规范化连续 Assistant 消息（沿用验证版的处理方式）
def merge_consecutive_assistants(messages):
    merged = []
    for message in messages:
        message = deepcopy(message)
        if merged and message["role"] == "assistant" and merged[-1]["role"] == "assistant":
            prev = merged[-1]
            first = prev.get("content") or ""
            second = message.get("content") or ""
            if second:
                prev["content"] = (first + "\n" + second).strip() if first else second
            if message.get("tool_calls"):
                prev.setdefault("tool_calls", []).extend(message["tool_calls"])
        else:
            merged.append(message)
    return merged


def iter_sft_examples(messages, source_file):
    history = []
    has_user = False
    for index, message in enumerate(merge_consecutive_assistants(messages)):
        if message["role"] == "assistant" and has_user:
            yield {
                "source_file": source_file,
                "target_index": index,
                "prompt_messages": history.copy(),
                "target_message": message,
            }
        history.append(message)
        if message["role"] == "user":
            has_user = True

## 5. Chat Template 与 Prompt / Completion 渲染

使用 Gemma 4 的 Chat Template 和同一组 Agent 工具定义，保证训练与推理的消息格式一致。裁剪保留最近的**完整 User 轮次**；对少数 Chat Template 在轮次结尾插入 `<turn|>` 的情况进行严格前缀对齐，其余不一致的样本计入过滤统计。

In [9]:
# 优先加载已经下载到本地的 Tokenizer
source = str(TOKENIZER_DIR) if (TOKENIZER_DIR / "tokenizer_config.json").exists() else MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(source)
tokenizer.padding_side = "right"
print("Tokenizer:", source)

Tokenizer: sft-data/gemma4-tokenizer


In [10]:
from typing import Any

# Agent工具定义
KNOWN_TOOL_SCHEMAS: dict[str, dict[str, Any]] = {
    "bash": {
        "description": "Run a shell command in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "Shell command to run."},
                "cmd": {"type": "string", "description": "Shell command to run."},
                "timeout": {"type": "number", "description": "Optional timeout in milliseconds."},
            },
            "required": [],
        },
    },
    "read": {
        "description": "Read a file or image from the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to read."},
                "file": {"type": "string", "description": "Path to read."},
                "offset": {"type": "number", "description": "Optional starting line."},
                "limit": {"type": "number", "description": "Optional line limit."},
                "start": {"type": "number", "description": "Optional starting line."},
                "end": {"type": "number", "description": "Optional ending line."},
            },
            "required": [],
        },
    },
    "edit": {
        "description": "Edit a file in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to edit."},
                "oldText": {"type": "string", "description": "Text to replace."},
                "newText": {"type": "string", "description": "Replacement text."},
                "edits": {"type": "array", "description": "Structured edits."},
                "patch": {"type": "string", "description": "Patch content."},
            },
            "required": [],
        },
    },
    "write": {
        "description": "Write content to a file in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to write."},
                "content": {"type": "string", "description": "File content."},
            },
            "required": [],
        },
    },
    "grep": {
        "description": "Search text in files.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Search pattern."},
                "path": {"type": "string", "description": "Path to search."},
                "limit": {"type": "number", "description": "Optional result limit."},
                "literal": {"type": "boolean", "description": "Treat pattern literally."},
                "context": {"type": "number", "description": "Context lines."},
            },
            "required": [],
        },
    },
    "find": {
        "description": "Find files or text in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Pattern to find."},
                "path": {"type": "string", "description": "Path to search."},
                "limit": {"type": "number", "description": "Optional result limit."},
            },
            "required": [],
        },
    },
    "ls": {
        "description": "List files in a directory.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Directory path."},
                "limit": {"type": "number", "description": "Optional result limit."},
            },
            "required": [],
        },
    },
    "todo": {
        "description": "Manage a lightweight task list.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "description": "Task-list action."},
                "text": {"type": "string", "description": "Task text."},
                "id": {"type": "string", "description": "Task identifier."},
            },
            "required": [],
        },
    },
}

In [11]:
# 生成默认工具定义
def generic_tool_schema(name: str) -> dict[str, Any]:
    return {
        "description": f"Pi coding-agent tool named {name}.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    }

# 生成工具列表
def make_tool_definitions(tool_names: set[str]) -> list[dict[str, Any]]:
    tools: list[dict[str, Any]] = []
    for name in sorted(n for n in tool_names if n):
        schema = KNOWN_TOOL_SCHEMAS.get(name, generic_tool_schema(name))
        tools.append({"type": "function", "function": {"name": name, **schema}})
    return tools

In [12]:
# 沿用验证版的八种 Agent 工具定义
TOOLS = make_tool_definitions(set(KNOWN_TOOL_SCHEMAS))

# 将消息按用户轮次分割
def split_into_user_turns(history):
    turns = []
    for msg in history:
        if msg["role"] == "user":
            turns.append([msg])
        elif turns:
            turns[-1].append(msg)
    return turns

# 验证已保留的历史中，工具调用必须有匹配的工具结果
def validate_tool_pairs(messages):
    pending = Counter()
    for msg in messages:
        if msg["role"] == "assistant":
            for call in msg.get("tool_calls", []):
                pending[call["id"]] += 1
        elif msg["role"] == "tool":
            key = msg.get("tool_call_id")
            if not key or pending[key] <= 0:
                return False
            pending[key] -= 1
    return all(n == 0 for n in pending.values())

In [13]:
# 渲染消息
def render_messages(messages, tokenizer, tools):
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

# 渲染完整对话
def render_full_conversation(messages, tokenizer, tools):
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

# 计算token数
def count_tokens(text, tokenizer):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [14]:
class SFTDataError(ValueError):
    pass

# 返回严格对齐的 Prompt 与完整序列 Token IDs
def render_aligned_pair(messages, target, tokenizer, tools):
    prompt_text = render_messages(messages, tokenizer, tools)
    full_text = render_full_conversation(messages + [target], tokenizer, tools)

    if full_text.startswith(prompt_text):
        pass
    elif (
        prompt_text.endswith("<turn|>\n")
        and full_text.startswith(prompt_text[:-len("<turn|>\n")])
    ):
        # Gemma 4 将后续工具调用接在前一模型轮次中
        prompt_text = prompt_text[:-len("<turn|>\n")]
    else:
        raise SFTDataError("chat_template_prefix_mismatch")

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise SFTDataError("token_prefix_mismatch")
    return prompt_ids, full_ids

# 按完整 User 轮次尝试扩展，直接返回可训练的 Token 级样本。
def build_tokenized_sft_example(example, tokenizer, tools, max_length=2048):
    history = example["prompt_messages"]
    target = example["target_message"]
    turns = split_into_user_turns(history)
    if not turns or target["role"] != "assistant":
        return None, "missing_user_or_target"

    selected = []
    selected_ids = None
    for turn in reversed(turns):
        candidate = turn + selected
        # 完整轮次截取后，也不能留下孤立工具返回
        if not validate_tool_pairs(candidate):
            return None, "unpaired_tool_result"

        try:
            prompt_ids, full_ids = render_aligned_pair(candidate, target, tokenizer, tools)
        except SFTDataError as exc:
            return None, str(exc)

        if len(full_ids) <= max_length:
            selected = candidate
            selected_ids = (prompt_ids, full_ids)
        else:
            break

    if selected_ids is None:
        return None, "overlength_latest_turn"

    prompt_ids, full_ids = selected_ids
    if len(prompt_ids) >= len(full_ids):
        return None, "empty_completion"

    # 当前 target 的工具名必须可被当前工具集表示
    unknown_tools = [
        call["function"]["name"]
        for call in target.get("tool_calls", [])
        if call["function"]["name"] not in KNOWN_TOOL_SCHEMAS
    ]
    if unknown_tools:
        return None, "unsupported_target_tool"

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):],
    }, None

## 6. 按 Session 划分数据并构建 Hugging Face Dataset
**先分 Session，再生成训练样本**，避免同一个执行轨迹的重叠历史同时进入训练集和验证集。逐 Session 将 Token 数据写入 JSONL，最后载入 Hugging Face Dataset；不在内存中堆积全部展开的历史样本。

In [15]:
# 固定随机种子，以 Session 文件为单位进行 90% / 10% 划分
shuffled_files = list(raw_files)
random.Random(SEED).shuffle(shuffled_files)

val_count = max(1, round(len(shuffled_files) * EVAL_RATIO))
val_files = set(shuffled_files[:val_count])
train_files = set(shuffled_files[val_count:])

print("Train Session 数:", len(train_files))
print("Validation Session 数:", len(val_files))

Train Session 数: 571
Validation Session 数: 63


In [16]:
# 流式构建全量 Token 级 SFT 数据；每次只处理一个 Session
TRAIN_JSONL = DATA_DIR / f"train_tokens_{MAX_LENGTH}.jsonl"
VAL_JSONL = DATA_DIR / f"validation_tokens_{MAX_LENGTH}.jsonl"


def build_full_token_files(files, train_files, tokenizer, tools):
    stats = Counter()
    # 使用临时文件，避免中途失败留下看似完整的缓存
    train_tmp = TRAIN_JSONL.with_suffix(".jsonl.tmp")
    val_tmp = VAL_JSONL.with_suffix(".jsonl.tmp")

    with train_tmp.open("w", encoding="utf-8") as train_f, \
         val_tmp.open("w", encoding="utf-8") as val_f:
        for file_index, path in enumerate(files, start=1):
            stats["session_files"] += 1
            try:
                events = load_session(path)
                stats["raw_events"] += len(events)
                messages = convert_session_messages(events)
            except (OSError, UnicodeError, json.JSONDecodeError, ValueError) as exc:
                stats["failed_sessions"] += 1
                print(f"跳过损坏 Session: {path.name} ({exc})")
                continue

            dest = train_f if path in train_files else val_f
            partition = "train" if path in train_files else "validation"

            for example in iter_sft_examples(messages, path.name):
                stats["candidate_examples"] += 1
                try:
                    row, reason = build_tokenized_sft_example(
                        example, tokenizer, tools, max_length=MAX_LENGTH
                    )
                except (TypeError, KeyError, ValueError) as exc:
                    row, reason = None, "invalid_message_or_template"

                if row is None:
                    stats[f"dropped_{reason}"] += 1
                    continue

                dest.write(json.dumps(row, ensure_ascii=False) + "\n")
                stats[f"{partition}_examples"] += 1

            if file_index % 25 == 0 or file_index == len(files):
                print(
                    f"Session {file_index}/{len(files)} | "
                    f"train={stats['train_examples']} | "
                    f"validation={stats['validation_examples']} | "
                    f"dropped={sum(v for k, v in stats.items() if k.startswith('dropped_'))}",
                    flush=True,
                )

    train_tmp.replace(TRAIN_JSONL)
    val_tmp.replace(VAL_JSONL)
    return dict(stats)

# conversion_stats = build_full_token_files(raw_files, train_files, tokenizer, TOOLS)
# (OUTPUT_DIR / "conversion_stats.json").write_text(
#     json.dumps(conversion_stats, indent=2, ensure_ascii=False), encoding="utf-8"
# )
# print("样本转换统计:", conversion_stats)

Session 25/634 | train=267 | validation=82 | dropped=607
Session 50/634 | train=529 | validation=82 | dropped=1089
Session 75/634 | train=676 | validation=118 | dropped=1395
Session 100/634 | train=915 | validation=147 | dropped=1751
Session 125/634 | train=1060 | validation=151 | dropped=1910
Session 150/634 | train=1169 | validation=162 | dropped=2130
Session 175/634 | train=1363 | validation=165 | dropped=2512
Session 200/634 | train=1578 | validation=171 | dropped=2898
Session 225/634 | train=1756 | validation=178 | dropped=3187
Session 250/634 | train=1920 | validation=195 | dropped=3481
Session 275/634 | train=2237 | validation=229 | dropped=3999
Session 300/634 | train=2435 | validation=229 | dropped=4472
Session 325/634 | train=2568 | validation=305 | dropped=4960
Session 350/634 | train=2765 | validation=324 | dropped=5426
Session 375/634 | train=2893 | validation=346 | dropped=5728
Session 400/634 | train=3146 | validation=348 | dropped=6273
Session 425/634 | train=3332 | val

In [17]:
from datasets import load_dataset

# JSONL -> Arrow Dataset；Train / Validation 分别加载
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_JSONL),
        "validation": str(VAL_JSONL),
    },
)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

if len(train_dataset) == 0 or len(eval_dataset) == 0:
    raise RuntimeError("训练集或验证集没有有效样本；请查看 conversion_stats.json")

print("训练样本:", len(train_dataset))
print("验证样本:", len(eval_dataset))
print(train_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

训练样本: 4770
验证样本: 534
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4770
})


In [18]:
# 自定义Data Collator，将同一批次的数据填充到同一长度
import torch

class SFTDataCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

        # 如果 Tokenizer 没有定义 PAD，则使用 EOS 作为填充值
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id

        if self.pad_token_id is None:
            raise ValueError("Tokenizer 没有可用的 PAD/EOS Token")

    def __call__(self, features):
        # 当前 Batch 中最长序列
        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            input_ids = feature["input_ids"]
            attention_mask = feature["attention_mask"]
            labels = feature["labels"]

            assert len(input_ids) == len(attention_mask) == len(labels)

            pad_length = max_length - len(input_ids)

            # 右侧 Padding
            batch_input_ids.append(
                input_ids
                + [self.pad_token_id] * pad_length
            )

            batch_attention_mask.append(
                attention_mask
                + [0] * pad_length
            )

            batch_labels.append(
                labels
                + [-100] * pad_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids, dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                batch_attention_mask, dtype=torch.long
            ),
            "labels": torch.tensor(
                batch_labels, dtype=torch.long
            )
        }

data_collator = SFTDataCollator(tokenizer)

## 7. 加载 Gemma 4 E2B 并配置 LoRA
本节沿用 BF16 加载和 `r=8, lora_alpha=16`，将 LoRA 从仅 Attention 的 `q_proj/v_proj` 扩展为 **7 类投影层**：Attention 的 `q/k/v/o_proj` 与 MLP 的 `gate/up/down_proj`。使用正则表达式只匹配语言模型层，避免对视觉、音频分支添加 Adapter。

In [20]:
from transformers import AutoModelForMultimodalLM
from peft import LoraConfig, TaskType, get_peft_model
import re

# 本 Notebook 默认只使用一张 GPU；避免 Trainer 意外使用 DataParallel
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        "请在启动 Notebook Kernel 之前设置 CUDA_VISIBLE_DEVICES=0，"
        "确保只看见一张 RTX 5090。"
    )

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

LORA_TARGET_PATTERN = (
        r".*language_model\.layers\.\d+\."
        r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)"
        r"|mlp\.(gate_proj|up_proj|down_proj))$"
    )

# 先在真实模型结构中进行完全匹配，再将模块全名传给 PEFT，比直接使用 "q_proj" 等短名称更容易确认没有误选非语言模块。
pattern = re.compile(LORA_TARGET_PATTERN)
lora_targets = [
    name for name, module in model.named_modules()
    if pattern.fullmatch(name)
]

if not lora_targets:
    raise RuntimeError("未匹配到 LoRA 目标模块，请检查 Gemma 4 模型结构")

print("LoRA 目标模块数量:", len(lora_targets))
print("前 10 个目标模块:")
for name in lora_targets[:10]:
    print(" -", name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=lora_targets,
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

LoRA 目标模块数量: 245
前 10 个目标模块:
 - model.language_model.layers.0.self_attn.k_proj
 - model.language_model.layers.0.self_attn.q_proj
 - model.language_model.layers.0.self_attn.v_proj
 - model.language_model.layers.0.self_attn.o_proj
 - model.language_model.layers.0.mlp.gate_proj
 - model.language_model.layers.0.mlp.up_proj
 - model.language_model.layers.0.mlp.down_proj
 - model.language_model.layers.1.self_attn.k_proj
 - model.language_model.layers.1.self_attn.q_proj
 - model.language_model.layers.1.self_attn.v_proj
trainable params: 12,668,928 || all params: 5,135,846,944 || trainable%: 0.2467


## 8. 配置 SFT 训练参数与 Trainer
使用 Hugging Face `Trainer` 直接训练第 6 节构建的 `input_ids / attention_mask / labels`，不重复 Tokenization。`labels=-100` 的 Prompt 和 Padding 不参与 Loss。按 Epoch 在独立 Session 验证集上评估并保存最近的 Checkpoint。

In [21]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAIN_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
    optim="adamw_torch",
    max_grad_norm=TRAIN_CONFIG["max_grad_norm"],
    lr_scheduler_type="cosine",
    warmup_ratio=TRAIN_CONFIG["warmup_ratio"],
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_strategy="steps",
    logging_steps=TRAIN_CONFIG["logging_steps"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    prediction_loss_only=True,  # 验证时只计算 Loss，避免存储庞大的 logits
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)
# 自定义标签的 Loss 已由 Gemma 4 在模型内求平均；梯度累积无需重复传样本个数
trainer.model_accepts_loss_kwargs = False
print("Trainer 配置完成")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer 配置完成


## 9. 执行全量 SFT 训练、验证和保存 Adapter
全量训练会比单 Session 花费更长时间。Trainer 将每个 Epoch 进行验证并保存 Checkpoint；最终另外保存 LoRA Adapter、Tokenizer、指标和日志，便于复现与推理。

In [22]:
# 训练全部有效的 pi-mono 样本
train_result = trainer.train()
print("训练完成:", train_result.metrics)
print("Optimizer Steps:", trainer.state.global_step)

# 最终验证损失（验证集来自不同 Session）
eval_metrics = trainer.evaluate()
print("最终验证指标:", eval_metrics)

Epoch,Training Loss,Validation Loss
1,1.075878,1.181981
2,0.782757,1.028047
3,0.891977,1.026830


训练完成: {'train_runtime': 9795.562, 'train_samples_per_second': 1.461, 'train_steps_per_second': 0.365, 'total_flos': 2.6655626142233856e+17, 'train_loss': 1.108599709611793, 'epoch': 3.0}
Optimizer Steps: 3579


RuntimeError: on_train_begin must be called before on_evaluate

In [24]:
# 修复训练结束后单独调用 evaluate() 的 Notebook 回调异常
from transformers.utils.notebook import NotebookProgressCallback

# 只移除 Notebook 进度展示回调
if any(
    isinstance(cb, NotebookProgressCallback)
    for cb in trainer.callback_handler.callbacks
):
    trainer.remove_callback(NotebookProgressCallback)

print("已移除 NotebookProgressCallback")

# 重新评估，不需要重新训练
eval_metrics = trainer.evaluate()

print("最终验证指标:", eval_metrics)

已移除 NotebookProgressCallback
最终验证指标: {'eval_loss': 1.0268303155899048, 'eval_runtime': 54.5372, 'eval_samples_per_second': 9.791, 'eval_steps_per_second': 9.791, 'epoch': 3.0}


In [25]:
# 保存 Adapter、Tokenizer 与实验元数据
import pandas as pd

ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))

train_logs = pd.DataFrame(trainer.state.log_history)
train_logs.to_csv(OUTPUT_DIR / "train_logs.csv", index=False)

experiment_config = {
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "eval_ratio": EVAL_RATIO,
    "train_config": TRAIN_CONFIG,
    "train_examples": len(train_dataset),
    "validation_examples": len(eval_dataset),
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
    "conversion_stats": conversion_stats,
}
(OUTPUT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("Adapter 已保存:", ADAPTER_DIR)
print("训练日志已保存:", OUTPUT_DIR / "train_logs.csv")

Adapter 已保存: sft-output/gemma4-e2b-lora-full/final_adapter
训练日志已保存: sft-output/gemma4-e2b-lora-full/train_logs.csv
